In [ ]:
#!/usr/bin/env python3
"""
PE (h^-1) vs PF_area_used_km2: 1×2 panels with connected-bin medians
+ 95% bootstrap confidence intervals on the median (shaded)
Left  : PF_rainrate
Right : PF_maxrainrate
Independent y-limits for each panel; logarithmic area bins by default.
"""

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from collections import namedtuple
from scipy.stats import linregress

# ---------- USER PATHS ----------
basedir = '/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/precip_efficiency2/'
RR_CWP = basedir + "precip_efficiency_gt1hr_cwp_pf_rainrate_tb.nc"
RR_CLW = basedir + "precip_efficiency_gt1hr_clw_pf_rainrate_tb.nc"
RR_CIW = basedir + "precip_efficiency_gt1hr_ciw_pf_rainrate_tb.nc"
MX_CWP = basedir + "precip_efficiency_gt1hr_cwp_pf_maxrainrate_tb.nc"
MX_CLW = basedir + "precip_efficiency_gt1hr_clw_pf_maxrainrate_tb.nc"
MX_CIW = basedir + "precip_efficiency_gt1hr_ciw_pf_maxrainrate_tb.nc"

# ---------- SETTINGS ----------
X_VAR       = "PF_area_used_km2"   # x-axis variable in your files
USE_MEDIAN  = True                 # (kept, but we use median for bootstrap)
MIN_COUNT   = 5                    # min samples per bin to show a point
LOGX        = True                 # use log10-spaced bins + log x-axis
NBINS       = 14                   # number of x bins
INVERT_X    = False                # usually False for area
YLIM_LEFT   = (0, 40)              # left panel (PF_rainrate)
YLIM_RIGHT  = (0, 150)             # right panel (PF_maxrainrate)

# Range for regression in area (km^2)
X_MIN_REG = 1e3   # 10^3
X_MAX_REG = 2e4   # 2×10^4

# --- Bootstrap settings (median CI) ---
DO_BOOTSTRAP = True
NBOOT = 2000              # 1000–5000 typical
CI_PCT = (2.5, 97.5)      # 95% CI
SEED = 42

# --- Style ---
COLOR_CWP = "#6A5ACD"  # purple
COLOR_CLW = "#FFD60A"  # yellow
COLOR_CIW = "#00A7B7"  # teal
MS        = 80
ALPHA     = 0.95
TITLE_FS  = 22
LABEL_FS  = 22
TICK_FS_X = 16
TICK_FS_Y = 16
#CI_ALPHA  = 0.22        # shading opacity
CI_ALPHA  = 0.45        # shading opacity

Series = namedtuple("Series", "label path color")

# ---------- HELPERS ----------
def flatten_xy(ds, x_var=X_VAR):
    pe = ds["PRECEFF_TIMEAVG"].values.astype(float).ravel()   # h^-1
    x  = ds[x_var].values.astype(float).ravel()
    m  = np.isfinite(pe) & np.isfinite(x)
    return pe[m], x[m]

def auto_bins(x, nbins=NBINS, log=LOGX):
    x = x[np.isfinite(x)]
    if log:
        x = x[x > 0]
        lo, hi = np.nanpercentile(x, 1), np.nanpercentile(x, 99)
        lo = max(lo, 1e-2)
        if hi <= lo:
            hi = lo * 10
        return np.logspace(np.log10(lo), np.log10(hi), nbins + 1)
    else:
        lo, hi = np.nanpercentile(x, 1), np.nanpercentile(x, 99)
        if hi <= lo:
            hi = lo + 1.0
        return np.linspace(lo, hi, nbins + 1)

def binned_stat_bootstrap_median_ci(y, x, bins, min_count=1,
                                   nboot=2000, ci=(2.5, 97.5), seed=42):
    """
    Compute binned median and bootstrap CI for the median.
    Returns mids, median, lo, hi, counts.
    """
    rng = np.random.default_rng(seed)
    mids = 0.5 * (bins[:-1] + bins[1:])

    if np.any(np.diff(bins) <= 0):
        raise ValueError("Bin edges must be strictly increasing.")

    idx = np.digitize(x, bins) - 1

    med = np.full(mids.size, np.nan, dtype=float)
    lo  = np.full(mids.size, np.nan, dtype=float)
    hi  = np.full(mids.size, np.nan, dtype=float)
    cnt = np.zeros(mids.size, dtype=int)

    for i in range(mids.size):
        yi = y[idx == i]
        yi = yi[np.isfinite(yi)]
        cnt[i] = yi.size

        if yi.size < min_count:
            continue

        med[i] = np.nanmedian(yi)

        # bootstrap CI on median
        boot = np.empty(nboot, dtype=float)
        for b in range(nboot):
            samp = rng.choice(yi, size=yi.size, replace=True)
            boot[b] = np.nanmedian(samp)

        lo[i] = np.nanpercentile(boot, ci[0])
        hi[i] = np.nanpercentile(boot, ci[1])

    return mids, med, lo, hi, cnt

def binned_stat_simple(y, x, bins, min_count=1, use_median=True):
    """Fallback if you ever want non-bootstrap."""
    mids = 0.5 * (bins[:-1] + bins[1:])
    idx  = np.digitize(x, bins) - 1
    vals = np.full(mids.size, np.nan)
    cnt  = np.zeros(mids.size, dtype=int)
    for i in range(mids.size):
        yi = y[idx == i]
        yi = yi[np.isfinite(yi)]
        cnt[i] = yi.size
        if yi.size >= min_count:
            vals[i] = np.nanmedian(yi) if use_median else np.nanmean(yi)
    return mids, vals, cnt

def plot_series(ax, series, bins, ms, alpha, do_bootstrap=True):
    """
    Plot binned PE vs area for one series.
    Returns mids_used, stat_used (for regression), counts_used
    """
    with xr.open_dataset(series.path) as ds:
        pe, xv = flatten_xy(ds, X_VAR)

    if do_bootstrap:
        mids, med, lo, hi, cnt = binned_stat_bootstrap_median_ci(
            pe, xv, bins,
            min_count=MIN_COUNT,
            nboot=NBOOT,
            ci=CI_PCT,
            seed=SEED
        )
        m = np.isfinite(med)
        mids_u, med_u, lo_u, hi_u, cnt_u = mids[m], med[m], lo[m], hi[m], cnt[m]

        # CI shading
        ax.fill_between(mids_u, lo_u, hi_u, color=series.color, alpha=CI_ALPHA, linewidth=0)

        # Median points/line
        ax.scatter(mids_u, med_u, s=ms, c=series.color, edgecolor="k",
                   linewidths=1.5, alpha=alpha, label=series.label)
        ax.plot(mids_u, med_u, color=series.color, linewidth=1.6, alpha=0.95)

        return mids_u, med_u, cnt_u

    else:
        mids, stat, cnt = binned_stat_simple(pe, xv, bins, MIN_COUNT, USE_MEDIAN)
        m = np.isfinite(stat)
        mids_u, stat_u, cnt_u = mids[m], stat[m], cnt[m]
        ax.scatter(mids_u, stat_u, s=ms, c=series.color, edgecolor="k",
                   linewidths=1.5, alpha=alpha, label=series.label)
        ax.plot(mids_u, stat_u, color=series.color, linewidth=1.6, alpha=0.95)
        return mids_u, stat_u, cnt_u

def add_panel(ax, paths, panel_title, bins, ylim):
    """
    Add all ε series to an axis.
    Returns (stats_for_reg, series_list)
    where stats_for_reg is list of (mids, stat) in same order as series_list.
    """
    stats_for_reg = []
    series_list = []
    if paths.get("cwp"):
        series_list.append(Series(r"$\epsilon_{cwp}$", paths["cwp"], COLOR_CWP))
    if paths.get("clw"):
        series_list.append(Series(r"$\epsilon_{\ell}$", paths["clw"], COLOR_CLW))
    if paths.get("ciw"):
        series_list.append(Series(r"$\epsilon_{i}$", paths["ciw"], COLOR_CIW))

    for s in series_list:
        mids, stat, cnt = plot_series(ax, s, bins, MS, ALPHA, do_bootstrap=DO_BOOTSTRAP)
        stats_for_reg.append((mids, stat))

        # Optional: quick bin-sample diagnostic
        # print(s.label, "median bin count:", int(np.nanmedian(cnt)), "min:", int(np.nanmin(cnt)), "max:", int(np.nanmax(cnt)))

    if INVERT_X:
        ax.invert_xaxis()

    # Axes labels/style
    ax.set_xlabel(r"Area [$\mathrm{km}^2$]", fontsize=LABEL_FS)
    if LOGX:
        ax.set_xscale('log')

    ax.tick_params(axis="x", labelsize=TICK_FS_X)
    ax.tick_params(axis="y", labelsize=TICK_FS_Y)

    # remove top/right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.grid(False)
    ax.set_ylim(*ylim)
    ax.text(0.02, 0.96, panel_title, transform=ax.transAxes,
            ha="left", va="top", fontsize=20)

    return stats_for_reg, series_list

def fit_regression_in_range(ax, mids, stat,
                            x_min=X_MIN_REG, x_max=X_MAX_REG,
                            color="k", label=None):
    """
    Fit: y = a + b * log10(area)
    over x in [x_min, x_max], plot regression line, return (r, p).
    """
    m = (mids >= x_min) & (mids <= x_max)
    x = mids[m]
    y = stat[m]

    if x.size < 2:
        print(f"Not enough points for regression in range for {label}.")
        return np.nan, np.nan

    x_log = np.log10(x)
    slope, intercept, r_val, p_val, stderr = linregress(x_log, y)

    x_log_fit = np.linspace(np.log10(x_min), np.log10(x_max), 50)
    x_fit = 10.0**x_log_fit
    y_fit = intercept + slope * x_log_fit

    ax.plot(x_fit, y_fit, color=color, linestyle="--", linewidth=2)

    if label is not None:
        print(f"Regression ({label}) in log10(area) over [{x_min:.1e}, {x_max:.1e}] km^2:")
    else:
        print(f"Regression in log10(area) over [{x_min:.1e}, {x_max:.1e}] km^2:")
    print(f"   r = {r_val:.4f},  p = {p_val:.4e}")

    return r_val, p_val

# ---------- PLOT ----------
# Build bins from the left panel CWP file (robust to any one file)
with xr.open_dataset(RR_CWP) as _tmp:
    _, x_for_bins = flatten_xy(_tmp, X_VAR)
BINS = auto_bins(x_for_bins, NBINS, LOGX)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.4), dpi=130)

# Left: PF_rainrate
stats_rr, series_rr = add_panel(
    axes[0],
    {"cwp": RR_CWP, "clw": RR_CLW, "ciw": RR_CIW},
    r"$\bf{(a)}$ Mean rainrate",
    BINS,
    YLIM_LEFT
)
axes[0].set_ylabel(r"$\varepsilon$ [h$^{-1}$]", fontsize=LABEL_FS)

# Right: PF_maxrainrate
stats_mx, series_mx = add_panel(
    axes[1],
    {"cwp": MX_CWP, "clw": MX_CLW, "ciw": MX_CIW},
    r"$\bf{(b)}$ Max-rainrate",
    BINS,
    YLIM_RIGHT
)

# --- Fit regression between 1e3 and 2e4 km^2 (in log10 space) for each ε metric ---
print("\n=== PF_rainrate panel (log10(area) 1e3–2e4 km^2) ===")
for (mids, stat), s in zip(stats_rr, series_rr):
    fit_regression_in_range(
        axes[0], mids, stat,
        x_min=X_MIN_REG, x_max=X_MAX_REG,
        color=s.color, label=f"Rainrate {s.label}"
    )

print("\n=== PF_maxrainrate panel (log10(area) 1e3–2e4 km^2) ===")
for (mids, stat), s in zip(stats_mx, series_mx):
    fit_regression_in_range(
        axes[1], mids, stat,
        x_min=X_MIN_REG, x_max=X_MAX_REG,
        color=s.color, label=f"Max-rainrate {s.label}"
    )

# One legend on the right panel
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles, labels, frameon=False, loc="center left", fontsize=20)

# Cosmetic: rotate x-ticks slightly for readability (for log this is optional)
for ax in axes:
    if not LOGX:
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

plt.tight_layout(w_pad=1.2)

# ---- Save ----
fig.savefig('/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure7_PE_extent.png', dpi=200, bbox_inches='tight')
fig.savefig('/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure7_PE_extent.pdf', dpi=200, bbox_inches='tight')

